# 6th attempt - RCNN - 5

In [55]:
import numpy as np
import pandas as pd
from functions import *
from read_from_file_df import *
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [56]:
SIZE = 5
AMOUNT_BOARDS = 10000

In [57]:
gen = 2
name_df = f'{PATH_DF}\\{SIZE}-{AMOUNT_BOARDS}\\{SIZE}size_{AMOUNT_BOARDS}boards_{gen}gen_reverse'
reverse_df = pd.read_pickle(f'{name_df}.pkl')

In [58]:
new_columns = [f'Col_{i}' for i in range(gen*SIZE*SIZE)]
reverse_df_sort = reverse_df.sort_values(by = new_columns).reset_index(drop=True)
for i in reverse_df_sort.columns:
    reverse_df_sort[i] = reverse_df_sort[i].astype(int)

In [59]:
print("reverse df:", len(reverse_df))
print("reverse df sort:",len(reverse_df_sort))

reverse df: 29760
reverse df sort: 29760


In [60]:
# Step 1: Prepare Data
amount_features = len(reverse_df_sort.columns) - SIZE*SIZE #the previous boards
features = reverse_df_sort.iloc[:, :amount_features]
name_col = 'Col_' + str(amount_features + 1)  # Target: the first pixel in the board
target = reverse_df_sort[name_col]

# Step 2: Split Data
X_train_val, X_test, y_train_val, y_test = train_test_split(features, target, test_size=0.1, random_state=365)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1, random_state=365)

print("len x train: ", len(X_train))
print("len x test: ",len(X_test))
print("len y train: ",len(y_train))
print("len y test: ",len(y_test))

len x train:  24105
len x test:  2976
len y train:  24105
len y test:  2976


In [61]:
X_train.shape

(24105, 25)

In [62]:
X_train_array = X_train.to_numpy()
y_train_array = y_train.to_numpy()

In [63]:
print(X_train_array.shape)
print(y_train_array.shape)

(24105, 25)
(24105,)


In [64]:
X_train_array = X_train_array.reshape((X_train.shape[0],SIZE,SIZE,1))
y_train_array = y_train_array.reshape((y_train.shape[0],1))

In [65]:
print(X_train_array.shape)
print(y_train_array.shape)

(24105, 5, 5, 1)
(24105, 1)


In [66]:
X_val_array = X_val.to_numpy()
X_val_array = X_val_array.reshape((X_val.shape[0],SIZE,SIZE,1))
y_val_array = y_val.to_numpy()
y_val_array = y_val_array.reshape((y_val.shape[0],1))

X_test_array = X_test.to_numpy()
X_test_array = X_test_array.reshape((X_test.shape[0],SIZE,SIZE,1))
y_test_array = y_test.to_numpy()
y_test_array = y_test_array.reshape((y_test.shape[0],1))

In [67]:
def build_RCNN5_softmax(gen_data, X_train_array, y_train_array, SIZE, batch_size=32, epochs=3):
    # --- פרמטרים ---
    gen_data = gen - 1    # מספר הלוחות הרציפים בקלט
    BATCH_SIZE = batch_size
    EPOCHS = epochs

    # --- PREPROCESSING ---
    # X_train_array.shape = (num_samples + gen_data, SIZE, SIZE, 1)
    # y_train_array.shape = (num_samples, 1)  ← תא אחד בלבד (0 או 1)

    num_samples = X_train_array.shape[0] - gen_data

    X_train = np.zeros((num_samples, gen_data, SIZE, SIZE, 1), dtype='float32')
    y_train = np.zeros((num_samples, 1), dtype='float32')  # רק תא אחד

    # # Convert labels to one-hot (for softmax)
    # y_train = tf.keras.utils.to_categorical(y_train, num_classes=2)
    # y_val   = tf.keras.utils.to_categorical(y_val,   num_classes=2)
    # y_test  = tf.keras.utils.to_categorical(y_test,  num_classes=2)

    for i in range(num_samples):
        X_train[i] = X_train_array[i:i+gen_data]   # רצף של gen_data לוחות
        y_train[i] = y_train_array[i]              # הפלט: תא אחד (0/1)

    print("X_train shape:", X_train.shape)  # (num_samples, gen_data, SIZE, SIZE, 1)
    print("y_train shape:", y_train.shape)  # (num_samples, 1)

    # --- MODEL ---
    model = tf.keras.Sequential([
        tf.keras.layers.ConvLSTM2D(
            filters=32,
            kernel_size=(3,3),
            activation='relu',
            padding='same',
            return_sequences=True,
            input_shape=(gen_data, SIZE, SIZE, 1)
        ),
        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.ConvLSTM2D(
            filters=64,
            kernel_size=(3,3),
            activation='relu',
            padding='same',
            return_sequences=False
        ),
        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(2, activation='softmax') 
    ])

In [68]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

NameError: name 'model' is not defined

In [ ]:
# אימון
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    shuffle=True
)

Epoch 1/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 27s 36ms/step - accuracy: 0.6539 - loss: 0.6418 - val_accuracy: 0.6687 - val_loss: 0.6269
Epoch 2/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 23s 37ms/step - accuracy: 0.6971 - loss: 0.5860 - val_accuracy: 0.6959 - val_loss: 0.5830
Epoch 3/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 38s 32ms/step - accuracy: 0.7056 - loss: 0.5687 - val_accuracy: 0.6905 - val_loss: 0.5963
Epoch 4/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 22s 34ms/step - accuracy: 0.7198 - loss: 0.5510 - val_accuracy: 0.6974 - val_loss: 0.5790
Epoch 5/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 24s 38ms/step - accuracy: 0.7392 - loss: 0.5281 - val_accuracy: 0.6951 - val_loss: 0.5925
Epoch 6/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 36s 30ms/step - accuracy: 0.7622 - loss: 0.4932 - val_accuracy: 0.6928 - val_loss: 0.6245
Epoch 7/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 23s 34ms/step - accuracy: 0.7861 - loss: 0.4597 - val_accuracy: 0.6795 - val_loss: 0.6659
Epoch 8/10
603/603 ━━━━━━━━━━━━━━━━━━━━ 23s 37ms/step - accuracy: 0.8074 - loss: 0.4138 - 

In [ ]:
X_test = X_test_array.reshape((-1, gen_data, SIZE, SIZE, 1)).astype('float32')
y_test = y_test_array.astype('float32')

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.3f}")

93/93 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6609 - loss: 0.9130
Test accuracy: 0.673


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_results(y_test, y_pred):
    """
    מצייר פיזור בין הפלט האמיתי לפלט החזוי.
    """
    y_test = y_test.flatten()
    y_pred = y_pred.flatten()

    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.3)
    
    # קו אידיאלי (תחזית = אמת)
    plt.plot([0, 1], [0, 1], linestyle='--')

    plt.xlabel("True Value (y_test)")
    plt.ylabel("Predicted Value (y_pred)")
    plt.title("Prediction Scatter Plot")
    plt.grid(True)
    plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

def evaluate_model(model, X_test_array, y_test_array, gen_data, SIZE):
    """
    Evaluate model with softmax outputs (2 classes).
    """

    # --- הכנת נתוני קלט ---
    X_test = X_test_array.reshape((-1, gen_data, SIZE, SIZE, 1)).astype('float32')

    # --- התאמת y_test ---
    # אם זה one-hot → להמיר למחלקות
    if len(y_test_array.shape) > 1 and y_test_array.shape[1] == 2:
        y_test = np.argmax(y_test_array, axis=1)
    else:
        # sparse labels (0/1)
        y_test = y_test_array.astype('int').reshape(-1)

    # --- חיזוי ---
    y_pred = model.predict(X_test)  # צורה: (n, 2)
    y_pred_classes = np.argmax(y_pred, axis=1)

    # --- Confusion Matrix ---
    cm = confusion_matrix(y_test, y_pred_classes)
    tn, fp, fn, tp = cm.ravel()

    acc = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)


    print("\n===== Evaluation Results =====")
    print("┌──────────────┬────────────┬────────────┐")
    print("│              │ Pred=Alive │ Pred=Dead  │")
    print("├──────────────┼────────────┼────────────┤")
    print(f"│ True=Alive   │ {tp:10d} │ {fn:10d} │")
    print(f"│ True=Dead    │ {fp:10d} │ {tn:10d} │")
    print("└──────────────┴────────────┴────────────┘")

    print("\n===== Classification Metrics =====")
    print(f"Accuracy : {acc:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall   : {recall:.3f}")
    print(f"F1-score : {f1:.3f}")

    return cm, acc, precision, recall, f1

In [ ]:
# יצירת סט בדיקה
num_samples_test = X_test_array.shape[0] - gen_data
X_test = np.zeros((num_samples_test, gen_data, SIZE, SIZE, 1), dtype='float32')
y_test = np.zeros((num_samples_test, 1), dtype='float32')

for i in range(num_samples_test):
    X_test[i] = X_test_array[i:i+gen_data]
    y_test[i] = y_test_array[i]

# הפעלת פונקציית ההערכה
results = evaluate_model(model, X_test, y_test, gen_data, SIZE)


93/93 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step

===== Evaluation Results =====
┌──────────────┬────────────┬────────────┐
│              │ Pred=Alive │ Pred=Dead  │
├──────────────┼────────────┼────────────┤
│ True=Alive   │        412 │        641 │
│ True=Dead    │        332 │       1590 │
└──────────────┴────────────┴────────────┘

===== Classification Metrics =====
Accuracy : 0.673
Precision: 0.554
Recall   : 0.391
F1-score : 0.459


In [ ]:
amount_features = len(reverse_df_sort.columns) - SIZE*SIZE #the previous boards
features = reverse_df_sort.iloc[:, :amount_features]
for i in range(SIZE*SIZE): # to any pixel in the expected board
    name_col = 'Col_' + str(i+amount_features)
    target = reverse_df_sort[name_col]
    X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.1, random_state=365)
    
    X_train_array = X_train.to_numpy()
    y_train_array = y_train.to_numpy()
    X_train_array = X_train_array.reshape((X_train.shape[0],SIZE,SIZE,1))
    y_train_array = y_train_array.reshape((y_train.shape[0],1))
    
    model = build_RCNN5_softmax(gen_data, X_train_array, y_train_array, SIZE, 32, 3)
    name_file = f"{PATH_MODELS}\\model6_RCNN_softmax\{SIZE}\\size_{SIZE}_pixel_{str(i+1)}.pkl"
    with open(name_file, 'wb') as file:
        pickle.dump(model, file)
    print(i)